# Notebook_04_Furusato_Provision_Complete_Workshop

**Edition:** v3.0.0-preview

Run one explicit stage at a time. Default is read-only preflight. Native Activator start/stop and correlated complete-file delivery remain separate gates. Run Notebook05 only after the three increments; it reads Lakehouse/Files, not Eventhouse.

Use the same private resource plan, gate and deployment-state evidence across stages. Environment identities are parameters, never embedded in this notebook. CLI deployment is the primary cross-machine orchestration path; this frontend uses normal NotebookUtils authentication.


In [ ]:
# Fabric parameter cell — environment identities remain in private JSON files.
PARTICIPANT_ID = "001"
ENVIRONMENT = ""  # Explicitly choose dev, test or prod.
EXPECTED_WORKSPACE_NAME = ""
SCOPE_FILE = ""  # Private scope JSON; use notebookutils.nbResPath + "/builtin/scope.json".
PLAN_FILE = ""  # Private resource-plan.json from preflight.
WRITE_GATE_FILE = ""  # Explicit folder mapping/write approval; never generated as approved.
PRIVATE_EVIDENCE_DIR = ""  # Outside the extracted source package; preserve between stages.
ACTION = "preflight"
APPLY_CHANGES = False
ALLOW_AUTOMATED_APPLY = False
CONFIRMED_PLAN_SHA256 = ""
RUN_NOTEBOOK01 = False
EXPECTED_DEFINITION_SHA256 = ""
RULE_STATEMENT_OVERRIDES = {}


In [ ]:
import base64, gzip, hashlib, json, sys, tempfile
from pathlib import Path
_PAYLOAD_CHUNKS = []


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_PAYLOAD_CHUNKS.append("...")


In [ ]:
_COMPRESSED = base64.b64decode("".join(_PAYLOAD_CHUNKS), validate=True)
if hashlib.sha256(_COMPRESSED).hexdigest() != "d66b321c6c2423211f3c17f98f96bcfb3650d07e725bbcf443ad46abb15b169d":
    raise ValueError("Embedded package fingerprint mismatch; no authentication attempted.")
_FILES = json.loads(gzip.decompress(_COMPRESSED))
_ROOT = Path(tempfile.gettempdir()) / "furusato-preview30-d66b321c6c242321"
for _relative, _encoded in _FILES.items():
    _path = (_ROOT / _relative).resolve()
    if _ROOT.resolve() not in _path.parents:
        raise ValueError("Unsafe embedded package path.")
    _raw = base64.b64decode(_encoded, validate=True)
    _path.parent.mkdir(parents=True, exist_ok=True)
    if _path.exists() and _path.read_bytes() != _raw:
        raise ValueError("Existing package bytes changed; use a clean driver session.")
    _path.write_bytes(_raw)
sys.path.insert(0, str(_ROOT / "tools/provisioning"))
for _module in ('preview30_runtime', 'preview30_deploy', 'preview30_verify', 'preview30_ontology', 'preview30_agent', 'preview30_activation', 'workshop_runtime', 'reference_kql', 'activation_runtime'):
    if _module in sys.modules and Path(sys.modules[_module].__file__).resolve().parent != (_ROOT / "tools/provisioning").resolve():
        del sys.modules[_module]
print("Sealed portable package loaded. No Fabric mutation has occurred.")


In [ ]:
if ACTION not in ('preflight', 'deploy-sources', 'verify-sources', 'deploy-realtime', 'deploy-core', 'handoff-ontology', 'verify-gold', 'deploy-semantic-model', 'refresh-model', 'verify-model', 'deploy-agent', 'publish-agent'):
    raise ValueError("Action is outside this notebook's bounded role. Use the CLI for importing notebooks.")
from pathlib import Path
from types import SimpleNamespace
import json
import shutil
import preview30_runtime as rt
import preview30_deploy as deploy
import preview30_verify as verify

context = notebookutils.runtime.context
if ENVIRONMENT not in ("dev", "test", "prod") or not SCOPE_FILE or not PRIVATE_EVIDENCE_DIR:
    raise ValueError("Set explicit environment, private scope file and private evidence directory. No cloud call made.")
scope_document = rt.load(Path(SCOPE_FILE))
scope = rt.scope_from_document(scope_document, ENVIRONMENT)
if scope["participantId"] != PARTICIPANT_ID:
    raise ValueError("Participant parameter differs from the private frozen scope.")
if scope["workspaceId"] != context["currentWorkspaceId"] or scope["workspaceName"] != context["currentWorkspaceName"]:
    raise ValueError("Notebook context differs from the private frozen workspace.")
if not EXPECTED_WORKSPACE_NAME or EXPECTED_WORKSPACE_NAME != context["currentWorkspaceName"]:
    raise ValueError("Set the exact expected workspace name before execution.")

class NotebookCredential:
    def get_token(self, scope_uri):
        audiences = {
            "https://api.fabric.microsoft.com/.default": "pbi",
            "https://storage.azure.com/.default": "storage",
            "https://kusto.kusto.windows.net/.default": "kusto",
            "https://analysis.windows.net/powerbi/api/.default": "pbi",
        }
        if scope_uri not in audiences:
            raise ValueError("Unsupported token audience.")
        return SimpleNamespace(token=notebookutils.credentials.getToken(audiences[scope_uri]))

credential = NotebookCredential()
evidence = rt.private_directory(Path(PRIVATE_EVIDENCE_DIR))
client = rt.EvidenceClient(scope, evidence, credential=credential)
current_notebook = client.request("GET", "/workspaces/" + scope["workspaceId"] + "/items/" + context["currentNotebookId"]).json()
if current_notebook.get("folderId") != scope["folderId"]:
    raise ValueError("The current Notebook is outside the explicitly selected folder.")

if ACTION == "preflight":
    if APPLY_CHANGES:
        raise ValueError("Preflight is read-only; APPLY_CHANGES must be False.")
    prefix = "/workspaces/" + scope["workspaceId"]
    inventory = {
        "workspace": client.request("GET", prefix).json(),
        "items": client.paged(prefix + "/items"),
        "folders": client.paged(prefix + "/folders"),
        "recoverableItems": client.paged(prefix + "/recoverableItems"),
    }
    baseline = rt.load(rt.PREVIEW / "provisioning/baseline-hashes.json")
    for relative, expected_sha in baseline["files"].items():
        path = rt.BASE / relative
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError("Extracted baseline byte mismatch: " + relative)
    state_file = evidence / "deployment-state.json"
    owned_state = rt.load(state_file) if state_file.exists() else None
    plan = rt.build_plan(scope, inventory, baseline, owned_state)
    rt.save(evidence / "resource-plan.json", plan)
    rt.save(evidence / "inventory-preflight.json", inventory)
    RESULT = {"mode": "read-only-preflight", "planSha256": plan["planSha256"],
              "blockers": plan["blockers"], "folderMappingVerified": scope["folderMappingVerified"],
              "cloudMutationPerformed": False}
else:
    read_only = ACTION.startswith("verify-") or ACTION == "preview-metadata"
    if not read_only and not APPLY_CHANGES:
        raise ValueError("No write made. Review the private plan, then explicitly enable APPLY_CHANGES.")
    if not read_only and context.get("isForPipeline", False) and not ALLOW_AUTOMATED_APPLY:
        raise ValueError("Automated mutation is not explicitly enabled.")
    if PLAN_FILE:
        external_plan = rt.load(Path(PLAN_FILE))
        rt.save(evidence / "resource-plan.json", external_plan)
    args = SimpleNamespace(
        environment=ENVIRONMENT, scope=Path(SCOPE_FILE), evidence_dir=evidence,
        write_gate=Path(WRITE_GATE_FILE) if WRITE_GATE_FILE else None,
        confirm=CONFIRMED_PLAN_SHA256, run_notebook01=RUN_NOTEBOOK01,
        expected_definition_sha256=EXPECTED_DEFINITION_SHA256,
        rule_statement_overrides=RULE_STATEMENT_OVERRIDES, credential=credential,
    )
    actions = {
        "deploy-sources": deploy.deploy_sources, "deploy-notebooks": deploy.deploy_notebooks,
        "verify-sources": verify.verify_sources, "deploy-realtime": deploy.deploy_realtime,
        "deploy-core": deploy.deploy_core, "preview-metadata": deploy.metadata_change,
        "apply-metadata": lambda a: deploy.metadata_change(a, apply=True),
        "handoff-ontology": deploy.handoff_ontology, "verify-gold": verify.verify_gold,
        "deploy-semantic-model": deploy.deploy_semantic_model, "verify-model": verify.verify_model,
        "refresh-model": deploy.refresh_model_once,
        "deploy-agent": deploy.deploy_agent,
        "publish-agent": deploy.publish_agent_candidate,
    }
    if ACTION not in actions:
        raise ValueError("Unsupported action; no private endpoint fallback.")
    RESULT = actions[ACTION](args)
print(json.dumps(RESULT, ensure_ascii=False, indent=2))
